# Fine-tune Bark Clean's decoder and joint

Upload a Murmur training export, fine-tune only the decoder and joint, gate on held-out and replay WER, and write a Core ML folder Murmur can install. See `README.md`.

In [ ]:
!pip -q install "transformers>=5.19" torch torchaudio coremltools "constriction>=0.5" soundfile soxr jiwer huggingface_hub
import os, json, zipfile, glob, shutil, hashlib, math, random
from google.colab import files
up = files.upload()
zip_name = next(iter(up))
shutil.rmtree("export", ignore_errors=True); os.makedirs("export")
zipfile.ZipFile(zip_name).extractall("export")
EXPORT = glob.glob("export/export-*")[0]
info = json.load(open(f"{EXPORT}/export.json")); print(info)

In [ ]:
from huggingface_hub import snapshot_download
REPO = snapshot_download("ConwayResearch/Underdog-Bark-3.0-Clean", allow_patterns=["model.bark", "bark_loader.py", "mlx/*"])
import sys; sys.path.insert(0, REPO)
import torch, bark_loader
model = bark_loader.load_model(f"{REPO}/model.bark", device="cuda", dtype=torch.float32)
from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained("nvidia/parakeet-tdt-0.6b-v3")
# Freeze the encoder only. encoder_projector, decoder.* and joint.* stay trainable.
for n, p in model.named_parameters():
    p.requires_grad = not n.startswith("encoder.")
print("trainable:", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
import soundfile as sf, soxr, numpy as np
def read(line):
    wav, sr = sf.read(f"{EXPORT}/{line['audio_filepath']}", dtype="float32", always_2d=True)
    wav = wav.mean(axis=1)
    if sr != 16000: wav = soxr.resample(wav, sr, 16000)
    return np.concatenate([np.zeros(4000, np.float32), wav])  # 0.25 s leading silence, as the model card uses
labelled = [json.loads(l) for l in open(f"{EXPORT}/manifest.jsonl") if l.strip()]
replay = [json.loads(l) for l in open(f"{EXPORT}/replay.jsonl") if l.strip()]
random.seed(0); random.shuffle(labelled)
n_test = max(10, len(labelled) // 10)
test, train_l = labelled[:n_test], labelled[n_test:]
assert train_l, "need more than %d labelled clips" % n_test
train = [(read(x), x["text"]) for x in train_l] + [(read(x), x["text"]) for x in replay]
test_set = [(read(x), x["text"]) for x in test]
replay_set = [(read(x), x["text"]) for x in replay[: max(10, len(replay) // 5)]]
print(len(train), "train clips;", len(test_set), "held-out;", len(replay_set), "replay eval")

In [ ]:
import jiwer
@torch.no_grad()
def transcribe(wavs):
    out = []
    for wav in wavs:
        inputs = processor(wav, sampling_rate=16000, return_tensors="pt").to("cuda")
        ids = model.generate(**inputs)
        out.append(processor.batch_decode(ids, skip_special_tokens=True)[0].strip())
    return out
norm = jiwer.Compose([jiwer.ToLowerCase(), jiwer.RemovePunctuation(), jiwer.RemoveMultipleSpaces(), jiwer.Strip()])
def wer(pairs):
    hyp = transcribe([w for w, _ in pairs]); ref = [t for _, t in pairs]
    return 100 * jiwer.wer(ref, hyp, truth_transform=norm, hypothesis_transform=norm)
model.eval()
base_test, base_replay = wer(test_set), wer(replay_set)
print(f"stock: held-out WER {base_test:.2f}  replay WER {base_replay:.2f}")

In [ ]:
from torch.utils.data import DataLoader
# transformers main, ParakeetProcessor.__call__: with text= it returns input_features,
# attention_mask, labels (token ids, padded with pad_token_id) and decoder_input_ids
# ([<blank>] + labels). ParakeetForTDT.forward needs decoder_input_ids whenever labels is
# given, and counts label lengths as (labels != config.pad_token_id), so labels are padded
# with pad_token_id and are NOT set to -100.
def collate(batch):
    wavs = [w for w, _ in batch]; texts = [t for _, t in batch]
    return processor(audio=wavs, text=texts, sampling_rate=16000, return_tensors="pt")
loader = DataLoader(train, batch_size=8, shuffle=True, collate_fn=collate)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-5, weight_decay=0.01)
best = (base_test, None); best_replay = None
for epoch in range(3):
    model.train()
    for step, batch in enumerate(loader):
        batch = {k: v.to("cuda") for k, v in batch.items()}
        loss = model(**batch).loss
        loss.backward(); opt.step(); opt.zero_grad()
        if step % 20 == 0: print(f"epoch {epoch} step {step} loss {loss.item():.3f}")
    model.eval()
    t, r = wer(test_set), wer(replay_set)
    print(f"epoch {epoch}: held-out WER {t:.2f}  replay WER {r:.2f}")
    if t < best[0]:
        best = (t, {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if not k.startswith("encoder.")})
        best_replay = r

In [ ]:
assert best[1] is not None, f"no epoch beat stock held-out WER {base_test:.2f}; nothing exported"
assert best_replay <= base_replay + 0.5, f"replay WER rose from {base_replay:.2f} to {best_replay:.2f}; nothing exported"
model.load_state_dict(best[1], strict=False)
print(f"exporting: held-out WER {best[0]:.2f} (stock {base_test:.2f}), replay WER {best_replay:.2f} (stock {base_replay:.2f})")

In [ ]:
import coremltools as ct
model.cpu().eval()
# Attribute names read from huggingface/transformers main at commit
# ca8c871ede62b521a75a2ddfc5e1968ac9c8e097 (modeling_parakeet.py, 2026-11-01):
#   ParakeetForRNNT.encoder_projector : Linear(encoder hidden 1024 -> 640)   (on the model, not the joint)
#   ParakeetRNNTDecoder.embedding     : Embedding(vocab_size, 640)
#   ParakeetRNNTDecoder.lstm          : LSTM(640, 640, num_layers=2, batch_first=True)
#   ParakeetRNNTDecoder.decoder_projector : Linear(640, 640), applied AFTER the LSTM
#   ParakeetRNNTJointNetwork.activation / .head : head = Linear(640, vocab_size + len(durations)) for TDT
# Split to match the stock Core ML contract (NeMo style): the Decoder model outputs the raw LSTM
# output, and the joint model owns both projections (decoder_projector and encoder_projector).
# The parity check below confirms this split reproduces the HF joint logits; if it fails, the
# stock Decoder output is the projected one and decoder_projector must move into DecoderWrap.
V = model.config.vocab_size  # 8193
class DecoderWrap(torch.nn.Module):
    def __init__(self, dec): super().__init__(); self.dec = dec
    def forward(self, targets, target_length, h_in, c_in):
        emb = self.dec.embedding(targets.long())             # [1,1,640]
        out, (h, c) = self.dec.lstm(emb, (h_in, c_in))       # [1,1,640], [2,1,640], [2,1,640]
        return out.transpose(1, 2), h, c                     # decoder [1,640,1], h_out, c_out
class JointWrap(torch.nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, encoder_step, decoder_step):
        enc = self.m.encoder_projector(encoder_step.transpose(1, 2))     # [1,1,640]
        dec = self.m.decoder.decoder_projector(decoder_step.transpose(1, 2))
        logits = self.m.joint.head(self.m.joint.activation(enc + dec))   # [1,1,V+durations]
        token_logits, dur_logits = logits[..., :V], logits[..., V:]
        top_logits, top_ids = torch.topk(token_logits, 64, dim=-1)       # [1,1,64]
        token_id = top_ids[..., :1].to(torch.int32)                      # argmax over tokens
        token_prob = top_logits[..., :1]                                 # max token logit, not softmaxed
        duration = dur_logits.argmax(-1, keepdim=True).to(torch.int32)
        return (token_id.reshape(1, 1, 1), token_prob.reshape(1, 1, 1), duration.reshape(1, 1, 1),
                top_ids.to(torch.int32).reshape(1, 1, 1, 64), top_logits.reshape(1, 1, 1, 64))
dec = DecoderWrap(model.decoder).eval()
jt = JointWrap(model).eval()

# Parity: the wrapper pair must reproduce the HF joint for one step.
with torch.no_grad():
    e = torch.randn(1, 1024, 1); tok = torch.tensor([[5]], dtype=torch.int32)
    d_raw, _, _ = dec(tok, torch.ones(1, dtype=torch.int32), torch.zeros(2, 1, 640), torch.zeros(2, 1, 640))
    ref = model.joint(decoder_hidden_states=model.decoder.decoder_projector(d_raw.transpose(1, 2)),
                      encoder_hidden_states=model.encoder_projector(e.transpose(1, 2)))
    tid, tprob, dur, _, _ = jt(e, d_raw)
    assert int(tid) == int(ref[..., :V].argmax()) and int(dur) == int(ref[..., V:].argmax()), "wrapper/HF mismatch"
    assert abs(float(tprob) - float(ref[..., :V].max())) < 1e-3

ex = (torch.zeros(1, 1, dtype=torch.int32), torch.ones(1, dtype=torch.int32), torch.zeros(2, 1, 640), torch.zeros(2, 1, 640))
dec_traced = torch.jit.trace(dec, ex)
dec_ml = ct.convert(dec_traced, convert_to="mlprogram", compute_precision=ct.precision.FLOAT16,
    minimum_deployment_target=ct.target.macOS15,
    inputs=[ct.TensorType(name="targets", shape=(1, 1), dtype=np.int32), ct.TensorType(name="target_length", shape=(1,), dtype=np.int32),
            ct.TensorType(name="h_in", shape=(2, 1, 640), dtype=np.float32), ct.TensorType(name="c_in", shape=(2, 1, 640), dtype=np.float32)],
    outputs=[ct.TensorType(name="decoder", dtype=np.float32), ct.TensorType(name="h_out", dtype=np.float32), ct.TensorType(name="c_out", dtype=np.float32)])
jt_traced = torch.jit.trace(jt, (torch.zeros(1, 1024, 1), torch.zeros(1, 640, 1)))
jt_ml = ct.convert(jt_traced, convert_to="mlprogram", compute_precision=ct.precision.FLOAT16,
    minimum_deployment_target=ct.target.macOS15,
    inputs=[ct.TensorType(name="encoder_step", shape=(1, 1024, 1), dtype=np.float32), ct.TensorType(name="decoder_step", shape=(1, 640, 1), dtype=np.float32)],
    outputs=[ct.TensorType(name="token_id", dtype=np.int32), ct.TensorType(name="token_prob", dtype=np.float32),
             ct.TensorType(name="duration", dtype=np.int32), ct.TensorType(name="top_k_ids", dtype=np.int32),
             ct.TensorType(name="top_k_logits", dtype=np.float32)])

In [ ]:
from coremltools.models.utils import compile_model
OUT = f"bark-decoder-{info['exportedAt'][:10]}"; shutil.rmtree(OUT, ignore_errors=True); os.makedirs(OUT)
dec_ml.save("Decoder.mlpackage"); jt_ml.save("JointDecisionv3.mlpackage")
for name in ["Decoder", "JointDecisionv3"]:
    compiled = compile_model(f"{name}.mlpackage")
    shutil.copytree(compiled, f"{OUT}/{name}.mlmodelc")
def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 22), b""): h.update(chunk)
    return h.hexdigest()
entries = []
for root, _, fs in os.walk(OUT):
    for f in fs:
        p = os.path.join(root, f); rel = os.path.relpath(p, OUT)
        entries.append({"path": rel, "size": os.path.getsize(p), "sha256": sha(p)})
manifest = {"trainedAt": info["exportedAt"][:10], "clips": len(train_l), "heldOutWER": round(best[0], 2),
            "replayWER": round(best_replay, 2), "baseRevision": info["barkRevision"], "files": entries}
json.dump(manifest, open(f"{OUT}/manifest.json", "w"), indent=2)
shutil.make_archive(OUT, "zip", ".", OUT)
files.download(f"{OUT}.zip")